## Purpose of this notebook is to develop a simple example of Monte Carlo simulation for long range capital planning.

Example Problem:
	
    Suppose a project costs $1 million and has average annual cash flows of $250,000/year with a standard of $100,000. If the cost of capital is %15, evaluate the project.

	If CFs are normally distributed then
		- 68% of the time $150k to $350k
		- 95% of the time $50k to $450k
		- 99% of the time -$50 to $550k

## The Basic Model

We should first build a model that handles non-variable inputs. Once that is working correctly, we can run Monte Carlo simulations using the model. In this model, we want to calculate Net Present Value (NPV) and Internal Rate of Return (IRR) using a mean cash flow (CF). Then later we can intergrate a standard deviation for the CF throughout the 10 year period.

### Capital Budget using NVP & IRR

1. Net Present Value (NPV)

#### What NPV tells you

**Net Present Value (NPV)** measures how much value a project adds (or destroys) in **today’s dollars**.
- **NPV > 0** → project creates value 
- **NPV = 0** → break-even versus required return 
- **NPV < 0** → project destroys value 

#### NPV Formula


$\text{NPV} = \sum_{t=0}^{T} \frac{CF_t}{(1 + r)^t}$


#### Where:

- $ {CF_t}$ = cash flow at time t
- r = discount rate (often WACC or hurdle rate)
- t = 0 is today (usually the initial investment, negative)

2. Internal Rate of Return (IRR)

#### What IRR tells you

**Internal Rate of Return (IRR)** is the **discount rate that makes NPV = 0**.

#### Decision rule:

- **IRR > WACC (or hurdle rate)** → accept 
- **IRR < WACC** → reject 

#### IRR Formula (Conceptual)

Solve for r:


$0 = \sum_{t=0}^{T} \frac{CF_t}{(1 + r)^t}$

In [12]:
# import dependencies
import numpy as np
import numpy_financial as npf

In [13]:
def npv(discount_rate, cash_flows):
    """
    Calculate Net Present Value (NPV).

    Parameters
    ----------
    discount_rate : float
        Discount rate (e.g., WACC or hurdle rate), expressed as a decimal.
    cash_flows : list or array
        Cash flows where index 0 is today (t=0).

    Returns
    -------
    float
        Net Present Value
    """
    cash_flows = np.array(cash_flows)
    periods = np.arange(len(cash_flows))
    return np.sum(cash_flows / (1 + discount_rate) ** periods)

In [14]:
# Calculate NPV using average cash flow
cash_flows = [-1000000,250000,250000,250000,250000,250000,250000,250000,250000,250000,250000]
discount_rate = 0.15

npv_value = npv(discount_rate, cash_flows)

npv_value

np.float64(254692.15646355768)

In [15]:
def irr(cash_flows):
    """
    Calculate Internal Rate of Return (IRR).

    Parameters
    ----------
    cash_flows : list or array
        Cash flows where index 0 is today (t=0).

    Returns
    -------
    float
        Internal Rate of Return
    """
    return npf.irr(cash_flows)

In [16]:
# Calculate IRR
irr(cash_flows)

0.21406465112705297


Using the average cash flow the NPV is greater then 0, so the project creates great value. Given  the Wacc of %15, which is less then our 21% IRR, we can again expect the project to add great value.

### Monte Carlo Simulation

We'll use the fact that the average CF is \$250,000 with a standard deviation of \$100,000 to compute our project risk using Monte Carlo simulation.

### Draws inputs from the distribution 

So here we are concerned with assigning the WACC random value from a distribution to calualte the CF over each year.

In [16]:
import random

mean_cf = 250000
stdev_cf = 100000

random.normalvariate(mean_cf, stdev_cf)

149672.04726984474

We can run the code multiple times to see it will choose a random number from our distribution each time, the value represents the respective years cash flow (CF).